# CSE 881 Project Imputation

In [15]:
import numpy as np
import pandas as pd

In [16]:
#load in data
data=pd.read_csv("wc_match_data.csv")
data

,match_id,year,stage,home_team,away_team,home_score,away_score,home_rank,away_rank,result,home_gdp,home_pop,away_gdp,away_pop
0,0,1930,Group Stage,France,Mexico,4.0,1.0,NaN,NaN,1,3.005906e+11,41610000.0,3.835178e+10,17175000.0
1,1,1930,Group Stage,Argentina,France,1.0,0.0,NaN,NaN,1,7.735969e+10,11896000.0,3.005906e+11,41610000.0
2,2,1930,Group Stage,Chile,Mexico,3.0,0.0,NaN,NaN,1,2.080102e+10,4266000.0,3.835178e+10,17175000.0
3,3,1930,Group Stage,Chile,France,1.0,0.0,NaN,NaN,1,2.080102e+10,4266000.0,3.005906e+11,41610000.0
4,4,1930,Group Stage,Argentina,Mexico,6.0,3.0,NaN,NaN,1,7.735969e+10,11896000.0,3.835178e+10,17175000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,924,2022,Quarterfinals,England,France,1.0,2.0,5.0,3.0,-1,2.615231e+12,68093380.0,2.671569e+12,68386540.0
925,925,2022,Semifinals,Argentina,Croatia,3.0,0.0,4.0,15.0,1,8.549143e+11,46736250.0,1.022846e+11,3790180.0
926,926,2022,Semifinals,France,Morocco,2.0,0.0,3.0,24.0,1,2.671569e+12,68386540.0,3.116550e+11,37101220.0
927,927,2022,Third Place Match,Croatia,Morocco,2.0,1.0,15.0,24.0,1,1.022846e+11,3790180.0,3.116550e+11,37101220.0


In [17]:
#show missingness
wc_missing = data[
    data[["home_gdp", "away_gdp", "home_pop", "away_pop"]]
    .isna()
    .any(axis=1)
]

wc_missing

,match_id,year,stage,home_team,away_team,home_score,away_score,home_rank,away_rank,result,home_gdp,home_pop,away_gdp,away_pop
13,13,1930,Group Stage,United States,Paraguay,3.0,0.0,NaN,NaN,1,1.322627e+12,123668000.0,NaN,880000.0
14,14,1930,Group Stage,Paraguay,Belgium,1.0,0.0,NaN,NaN,1,NaN,880000.0,6.409114e+10,8076000.0
19,19,1934,Round of 16,Hungary,Egypt,4.0,2.0,NaN,NaN,1,3.369598e+10,8919000.0,NaN,NaN
22,22,1934,Round of 16,Czechoslovakia,Romania,2.0,1.0,NaN,NaN,1,NaN,NaN,2.143920e+10,14924000.0
29,29,1934,Quarterfinals,Czechoslovakia,Switzerland,3.0,2.0,NaN,NaN,1,NaN,NaN,4.138914e+10,4140000.0
32,32,1934,Semifinals,Czechoslovakia,Germany,3.0,1.0,NaN,NaN,1,NaN,NaN,4.084154e+11,66409000.0
34,34,1934,Final,Italy,Czechoslovakia,2.0,1.0,NaN,NaN,1,1.879873e+11,42093000.0,NaN,NaN
42,42,1938,Round of 16,Czechoslovakia,Netherlands,3.0,0.0,NaN,NaN,1,NaN,NaN,7.267608e+10,8685000.0
48,48,1938,Quarterfinals,Brazil,Czechoslovakia,1.0,1.0,NaN,NaN,0,6.486564e+10,39480000.0,NaN,NaN
49,49,1938,Quarterfinals,Brazil,Czechoslovakia,2.0,1.0,NaN,NaN,1,6.486564e+10,39480000.0,NaN,NaN


In [18]:
#home data
home = data[["home_team", "year", "home_gdp", "home_pop"]].rename(columns={
    "home_team": "country",
    "home_gdp": "gdp",
    "home_pop": "pop"
})

#away data
away = data[["away_team", "year", "away_gdp", "away_pop"]].rename(columns={
    "away_team": "country",
    "away_gdp": "gdp",
    "away_pop": "pop"
})

#panel df, home and away separated
panel = pd.concat([home, away]).drop_duplicates()

## GDP and Population Imputation

In [19]:
#log scale for gdp and pop
panel["log_gdp"] = np.log(panel["gdp"])
panel["log_pop"] = np.log(panel["pop"])

In [20]:
#linear regression on log scale for gdp imputation
import statsmodels.formula.api as smf

gdp_model = smf.ols(
    "log_gdp ~ year + C(country)",
    data=panel.dropna(subset=["log_gdp"])
).fit()

missing_gdp = panel["log_gdp"].isna()

panel.loc[missing_gdp, "log_gdp"] = gdp_model.predict(panel[missing_gdp])
panel["gdp_imputed"] = np.exp(panel["log_gdp"])

In [21]:
#linear regression on log scale for population imputation
pop_model = smf.ols(
    "log_pop ~ year + C(country)",
    data=panel.dropna(subset=["log_pop"])
).fit()

missing_pop = panel["log_pop"].isna()

panel.loc[missing_pop, "log_pop"] = pop_model.predict(panel[missing_pop])
panel["pop_imputed"] = np.exp(panel["log_pop"])

In [22]:

#home merge
wc = data.merge(
    panel[["country", "year", "gdp_imputed", "pop_imputed"]],
    left_on=["home_team", "year"],
    right_on=["country", "year"],
    how="left"
)

wc["home_gdp"] = wc["home_gdp"].fillna(wc["gdp_imputed"])
wc["home_pop"] = wc["home_pop"].fillna(wc["pop_imputed"])

wc = wc.drop(columns=["country", "gdp_imputed", "pop_imputed"])

#away merge
wc = wc.merge(
    panel[["country", "year", "gdp_imputed", "pop_imputed"]],
    left_on=["away_team", "year"],
    right_on=["country", "year"],
    how="left"
)

wc["away_gdp"] = wc["away_gdp"].fillna(wc["gdp_imputed"])
wc["away_pop"] = wc["away_pop"].fillna(wc["pop_imputed"])

wc = wc.drop(columns=["country", "gdp_imputed", "pop_imputed"])

In [23]:
wc

,match_id,year,stage,home_team,away_team,home_score,away_score,home_rank,away_rank,result,home_gdp,home_pop,away_gdp,away_pop
0,0,1930,Group Stage,France,Mexico,4.0,1.0,NaN,NaN,1,3.005906e+11,41610000.0,3.835178e+10,17175000.0
1,1,1930,Group Stage,Argentina,France,1.0,0.0,NaN,NaN,1,7.735969e+10,11896000.0,3.005906e+11,41610000.0
2,2,1930,Group Stage,Chile,Mexico,3.0,0.0,NaN,NaN,1,2.080102e+10,4266000.0,3.835178e+10,17175000.0
3,3,1930,Group Stage,Chile,France,1.0,0.0,NaN,NaN,1,2.080102e+10,4266000.0,3.005906e+11,41610000.0
4,4,1930,Group Stage,Argentina,Mexico,6.0,3.0,NaN,NaN,1,7.735969e+10,11896000.0,3.835178e+10,17175000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924,924,2022,Quarterfinals,England,France,1.0,2.0,5.0,3.0,-1,2.615231e+12,68093380.0,2.671569e+12,68386540.0
925,925,2022,Semifinals,Argentina,Croatia,3.0,0.0,4.0,15.0,1,8.549143e+11,46736250.0,1.022846e+11,3790180.0
926,926,2022,Semifinals,France,Morocco,2.0,0.0,3.0,24.0,1,2.671569e+12,68386540.0,3.116550e+11,37101220.0
927,927,2022,Third Place Match,Croatia,Morocco,2.0,1.0,15.0,24.0,1,1.022846e+11,3790180.0,3.116550e+11,37101220.0


## FIFA Rank Imputation

In [24]:

#elo algorithm

#get teams 
teams = pd.unique(wc[["home_team", "away_team"]].values.ravel())
#init elo score
elo = {team: 1500 for team in teams}

#expected score function
def expected_score(r1, r2):
    return 1 / (1 + 10 ** ((r2 - r1) / 400))

#update elo function
def update_elo(r1, r2, score1, K=20):
    exp1 = expected_score(r1, r2)
    return r1 + K * (score1 - exp1)

#sort matches by year
wc = wc.sort_values("year").reset_index(drop=True)

#fill na
wc["home_elo"] = np.nan
wc["away_elo"] = np.nan

#elo updates
for i, row in wc.iterrows():
    home = row["home_team"]
    away = row["away_team"]

    r_home = elo[home]
    r_away = elo[away]

    #pre-match ratings
    wc.at[i, "home_elo"] = r_home
    wc.at[i, "away_elo"] = r_away

    #match result
    if row["home_score"] > row["away_score"]:
        s_home, s_away = 1, 0
    elif row["home_score"] < row["away_score"]:
        s_home, s_away = 0, 1
    else:
        s_home, s_away = 0.5, 0.5

    # adjust k 
    goal_diff = abs(row["home_score"] - row["away_score"])
    K = 20 + goal_diff * 5

    #update ratings
    elo[home] = update_elo(r_home, r_away, s_home, K)
    elo[away] = update_elo(r_away, r_home, s_away, K)


#elo long
elo_long = pd.concat([
    wc[["year", "home_team", "home_elo"]].rename(columns={
        "home_team": "team",
        "home_elo": "elo"
    }),
    wc[["year", "away_team", "away_elo"]].rename(columns={
        "away_team": "team",
        "away_elo": "elo"
    })
])


#full panel
years = sorted(wc["year"].unique())
teams = pd.unique(elo_long["team"])

full_index = pd.MultiIndex.from_product([teams, years], names=["team", "year"])

elo_full = (
    elo_long.sort_values("year")
    .groupby(["team", "year"])
    .last()
    .reindex(full_index)
    .groupby(level=0)
    .ffill()
    .reset_index()
)

#calc ranking
elo_full["rank"] = elo_full.groupby("year")["elo"] \
    .rank(ascending=False, method="min")

#home merge
wc = wc.merge(
    elo_full[["team", "year", "rank"]],
    left_on=["home_team", "year"],
    right_on=["team", "year"],
    how="left"
).rename(columns={"rank": "home_rank_elo"}).drop(columns=["team"])

#away merge
wc = wc.merge(
    elo_full[["team", "year", "rank"]],
    left_on=["away_team", "year"],
    right_on=["team", "year"],
    how="left"
).rename(columns={"rank": "away_rank_elo"}).drop(columns=["team"])

#fill missing
wc["home_rank"] = wc["home_rank"].fillna(wc["home_rank_elo"])
wc["away_rank"] = wc["away_rank"].fillna(wc["away_rank_elo"])

#drop unwanted cols
wc = wc.drop(columns=["home_rank_elo", "away_rank_elo", "home_elo", "away_elo"], errors="ignore")


In [25]:
wc.describe()

,match_id,year,home_score,away_score,home_rank,away_rank,result,home_gdp,home_pop,away_gdp,away_pop
count,929.000000,929.000000,928.000000,928.000000,929.000000,929.000000,929.000000,9.290000e+02,9.290000e+02,9.290000e+02,9.290000e+02
mean,464.000000,1989.160388,1.786638,1.056034,16.684607,19.679225,0.314316,1.055272e+12,5.503161e+07,9.192392e+11,4.547282e+07
std,268.323499,24.520248,1.611633,1.068872,15.406090,15.867596,0.824085,1.809152e+12,7.172718e+07,2.071996e+12,7.910234e+07
min,0.000000,1930.000000,0.000000,0.000000,1.000000,1.000000,-1.000000,2.629770e+09,3.526000e+05,2.629770e+09,3.526000e+05
25%,232.000000,1970.000000,1.000000,0.000000,5.000000,8.000000,0.000000,1.191450e+11,1.039381e+07,7.407936e+10,8.489574e+06
50%,464.000000,1994.000000,1.500000,1.000000,12.000000,12.000000,1.000000,4.573775e+11,3.850179e+07,2.989814e+11,2.284525e+07
75%,696.000000,2010.000000,3.000000,2.000000,23.000000,29.000000,1.000000,1.391784e+12,6.772813e+07,1.105637e+12,5.633970e+07
max,928.000000,2022.000000,10.000000,7.000000,85.000000,85.000000,1.000000,1.949317e+13,1.275379e+09,1.949317e+13,1.275379e+09


In [26]:
#wc.to_csv("full_wc_match_data.csv", index=False)